<a href="https://colab.research.google.com/github/AI-Lab-2025-2-3rd/ai-project-32fyuabfa/blob/Once/%EB%8D%B0%EC%9D%B4%EC%BD%98_%EB%A9%9C%EB%A1%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

라이브러리

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import os
from sklearn.model_selection import train_test_split

!pip install catboost
from catboost import CatBoostClassifier, Pool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 19.6 MB/s eta 0:00:00


데이터셋: DACON 대회서 제공

In [ ]:
train = pd.read_csv('/content/train.csv')
test  = pd.read_csv('/content/test.csv')
print(train.head())

print("전체 개수:", len(df)) #13225개

FileNotFoundError: [Errno 2] No such file or directory: '/content/train.csv'

모델: CatBoost

참고 자료: [티스토리: CatBoost 알고리즘 설명](https://hyewon328.tistory.com/entry/CatBoost-CatBoost-%EC%95%8C%EA%B3%A0%EB%A6%AC%EC%A6%98%EC%97%90-%EB%8C%80%ED%95%9C-%EC%9D%B4%ED%95%B4)

# **CatBoost 알고리즘**
CatBoost는 범주형 변수로 이루어진 데이터셋에서 예측 성능이 우수함. Boosting 알고리즘의 경우 머신러닝의 Bias-Variance Tradeoff 문제 때문에 Overfitting이 발생할 가능성이 높아 이를 방지해주어야 함.

# **Catboost의 특징**
- Categorical feature: Overfitting 방지 위해 연관된 변수는 하나로 통일
- Ordered Target Encoding: data leakage 방지(정답 값을 미리 학습하는 현상. Overfitting 유발) 위해 *Laplace Smoothing* 방법 사용(저 방법이 뭔진 모르겠음). -> 데이터가 작을때 값이 치우치는 문제 완화
- Ordered Boosting: Random Permutation으로 값을 섞은 후 이전 시점의 데이터로만 오차를 계산해서 Overfitting 방지

In [ ]:
target = 'support_needs' #이렇게 하지 말고 train.drop(support_needs)로 X, Y에  넣기 -> 나중에 꼭 수정~~
id_col = 'ID'
feature_cols = [c for c in train.columns if c not in [id_col, target]]

X = train[feature_cols].copy()
y = train[target].copy()
X_test = test[feature_cols].copy()

# 컬럼 식별(object 타입?은 뭐지)
cat_features = [i for i, c in enumerate(X.columns) if X[c].dtype == 'object']

# 트레인, 테스트 분리하고
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Pool 구성
train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
valid_pool = Pool(X_va, y_va, cat_features=cat_features)
test_pool  = Pool(X_test, cat_features=cat_features)

# 모델 정의, 학습
model = CatBoostClassifier(
    loss_function='MultiClass',      # 멀티클래스(0/1/2)
    eval_metric='MultiClass',        # 기본 멀티클래스 지표
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    iterations=2000,                 # 충분히 큰 값 + 얼리스탑핑
    random_seed=42,
    verbose=200,
    early_stopping_rounds=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# ) 중요 변수 Top 확인(퍼센트)
fi = pd.DataFrame({
    'feature': X.columns,
    'importance': model.get_feature_importance(train_pool, type='FeatureImportance')ㅇ
}).sort_values('importance', ascending=False)
print(fi.head(10))

0:	learn: 1.0860820	test: 1.0870179	best: 1.0870179 (0)	total: 63.7ms	remaining: 2m 7s
200:	learn: 0.9375208	test: 0.9778030	best: 0.9765866 (87)	total: 2.78s	remaining: 24.9s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.9765866241
bestIteration = 87

Shrink model to first 88 iterations.
검증 예측(앞 20개): [2 2 0 0 0 2 2 2 2 2 0 0 0 0 0 0 2 0 0 0]
저장 완료 → /content/submission.csv
             feature  importance
0                age   31.269275
6    contract_length   23.480602
4   payment_interval   18.758016
7  after_interaction   11.114089
1             gender   10.420610
2             tenure    2.192621
3           frequent    1.406650
5  subscription_type    1.358138
